# Test filter_service and image_filters

In [1]:
import sys
import pandas as pd
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

Project directory set to: /home/dermascan/Dokumente/dermascan


In [3]:
import cv2
import numpy as np

from preprocessing import image_filters
from preprocessing import image_filters_opencv
from preprocessing.filter_service import FilterService
from dataset_manager import DatasetManager 

In [4]:
filter_service = FilterService(
    data_dir=str(proj_dir / "data")
)

dataset_manager = DatasetManager(
    data_dir=str(proj_dir / "data")
)

### Test applying and saving filters

#### Apply all filters for single image

In [ ]:
res = filter_service.apply_texture_filters(image_id="ISIC_0024306", filter_list=[image_filters.glcm_filter, image_filters.gabor_filter, image_filters.lbp_filter])
res['lbp_filter'].shape


In [ ]:
filter_service.save_applied_filters(res, image_id="ISIC_0024306")

#### Apply filters to multiple images and save them

In [5]:
dataset_manager.load_cleaned_metadata(
    filter_melanoma_only=True,
)
dataset_manager.df['dx'].value_counts()

dx
nv     6703
mel    1113
Name: count, dtype: int64

In [6]:
nv_samples = dataset_manager.get_n_samples(6703, lesion_type='nv')
mel_samples = dataset_manager.get_n_samples(1113, lesion_type='mel')

sample_df = pd.concat([nv_samples, mel_samples]) # concat
sample_df = sample_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

sample_df['dx'].value_counts()

dx
nv     6703
mel    1113
Name: count, dtype: int64

In [ ]:
result_map = filter_service.filtered_image_generator(
    list_image_ids=sample_df['image_id'].tolist(),
    filter_list=[image_filters.glcm_filter, image_filters.gabor_filter, image_filters.lbp_filter],
    batch_size=10,
    asnpz=False
)

## Refactoring der Filter Methoden mit OpenCV Bibliothek

In [5]:
dataset_manager.load_cleaned_metadata()

In [6]:
image = cv2.imread(dataset_manager.df[dataset_manager.df['image_id'] == 'ISIC_0033238']['image_path'].iloc[0], cv2.IMREAD_COLOR)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
image_gray_np = np.array(image_gray)

In [ ]:
image_gabor = image_filters.gabor_filter(image_gray_np)
image_gabor_cv = image_filters_opencv.gabor_filter(image_gray_np)

try:
    assert(image_gabor.shape == image_gabor_cv.shape)
except AssertionError:
    print(f"Shapes do not match! skimage: {image_gabor.shape}, OpenCV: {image_gabor_cv.shape}")

# Exact and tolerance-based value comparison
is_identical = np.array_equal(image_gabor, image_gabor_cv)
print(f"Exact match: {is_identical}")

is_close = np.allclose(image_gabor, image_gabor_cv, rtol=1e-5, atol=1e-8)
print(f"All close: {is_close}")

Exact match: True
All close: True


In [ ]:
image_glcm = image_filters.glcm_filter(image_gray_np)
image_glcm_cv = image_filters_opencv.glcm_filter(image_gray_np)

try:
    assert(image_glcm.shape == image_glcm_cv.shape)
except AssertionError:
    print(f"Shapes do not match! skimage: {image_glcm.shape}, OpenCV: {image_glcm_cv.shape}")

# Exact and tolerance-based value comparison
is_identical = np.array_equal(image_glcm, image_glcm_cv)
print(f"Exact match: {is_identical}")

is_close = np.allclose(image_glcm, image_glcm_cv, rtol=1e-5, atol=1e-8)
print(f"All close: {is_close}")